In [1]:
import os

import numpy as np
import pandas as pd

# 2020 Summer

# Control

In [2]:
DIRECTORY = './data/2020_S/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.dat')]
dataset_list.sort()

In [3]:
dataset_list

['CR1000_Table1_2020_05_06_21_19_15.dat',
 'CR1000_Table1_2020_05_13_21_19_15.dat',
 'CR1000_Table1_2020_05_25_14_35_46.dat',
 'CR1000_Table1_2020_06_10_16_29_39.dat',
 'CR1000_Table1_2020_06_18_15_51_17.dat',
 'CR1000_Table1_2020_07_13_15_19_58.dat']

In [4]:
greenhouse_df = []
for FILENAME in dataset_list:
    temp_df = pd.read_csv(DIRECTORY + FILENAME, sep=',', index_col='TIMESTAMP', skiprows=[0, 2, 3])
    greenhouse_df.append(temp_df)
greenhouse_df = pd.concat(greenhouse_df)

/home/phil/.virtualenvs/tf20/lib/python3.6/site-packages/ipykernel_launcher.py:5: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  """


In [5]:
greenhouse_df.index = pd.DatetimeIndex(greenhouse_df.index)

In [6]:
cr1000_df = greenhouse_df
cr1000_df = cr1000_df[['Loadcell_1', 'Loadcell_2', 'Pyrano_Wsec_1', 'Temp_Avg', 'Humid_Avg']]

In [7]:
date_index = pd.date_range(cr1000_df.index[0], cr1000_df.index[-1], freq='1 min')

In [8]:
cr1000_df = cr1000_df.reindex(date_index)

In [9]:
DIRECTORY = './data/2020_S/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.xlsx')]
dataset_list.sort()

In [10]:
greenhouse_df = []
divider = 0
for FILENAME in dataset_list:
    if 'A2' in FILENAME:
        divider += 1
    temp_df = pd.read_excel(DIRECTORY + FILENAME, index_col='date')
    temp_df.index = pd.DatetimeIndex(temp_df.index)
    temp_df = temp_df[['weight', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']]
    greenhouse_df.append(temp_df)

In [11]:
for i in range(len(greenhouse_df)):
    if greenhouse_df[i].index.round('1 min')[0].minute == 1:
        greenhouse_df[i].index = (greenhouse_df[i].index.round('1 min') - pd.Timedelta('1 min'))
    elif greenhouse_df[i].index.round('1 min')[0].minute == 2:
        greenhouse_df[i].index = (greenhouse_df[i].index.round('2 min') - pd.Timedelta('2 min'))
    else:
        greenhouse_df[i].index = greenhouse_df[i].index.round('1 min')

In [12]:
for i in range(len(greenhouse_df)):
    greenhouse_df[i] = greenhouse_df[i].groupby(greenhouse_df[i].index).first()

In [13]:
date_index = pd.date_range(greenhouse_df[0].index[0], greenhouse_df[-1].index[-1], freq='1 min')

In [14]:
_1 = pd.concat(greenhouse_df[:divider]).reindex(date_index)
_2 = pd.concat(greenhouse_df[divider:]).reindex(date_index)

In [15]:
_1.columns = ['Loadcell_3', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']
_2.columns = ['Loadcell_4', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']

In [16]:
_1

,Loadcell_3,subs_VWC,subs_EC,subs_bulk_EC,subs_temp,permittivity
2020-02-26 00:00:00,12.22,72.52,0.94,0.54,23.6,49.44
2020-02-26 00:01:00,12.22,72.52,0.94,0.54,23.6,49.44
2020-02-26 00:02:00,12.22,72.53,0.93,0.54,23.6,49.45
2020-02-26 00:03:00,12.22,72.49,0.95,0.55,23.6,49.40
2020-02-26 00:04:00,12.22,72.39,0.95,0.54,23.6,49.30
...,...,...,...,...,...,...
2020-07-09 23:55:00,-0.24,8.97,0.00,0.00,26.8,2.21
2020-07-09 23:56:00,-0.24,8.97,0.00,0.00,26.8,2.21
2020-07-09 23:57:00,-0.24,9.05,0.00,0.00,26.8,2.23
2020-07-09 23:58:00,-0.25,8.94,0.00,0.00,26.8,2.21


In [17]:
SW2_df = pd.concat([_1.loc[pd.date_range(cr1000_df.index[0], _1.index[-1], freq='1 min')],
                    cr1000_df.loc[pd.date_range(cr1000_df.index[0], _1.index[-1], freq='1 min')]], axis=1)

In [18]:
SW2_df.columns = ['loadcell_3', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_1', 'loadcell_2', 'rad', 'temp', 'hum']

In [19]:
SW2_df = SW2_df[['temp', 'hum', 'rad', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_1', 'loadcell_2', 'loadcell_3']]

In [20]:
SW2_df['temp'] = ((SW2_df['temp']-1000)/80)
SW2_df['hum'] = ((SW2_df['hum']-1000)/40)
SW2_df['loadcell_1'] = ((SW2_df['loadcell_1']/100))
SW2_df['loadcell_2'] = ((SW2_df['loadcell_2']/100))

In [21]:
SW2_df.to_csv('./results/2020_S/SW2_greenhouse_origin.csv')

# Far-red

In [22]:
cr1000_a3_df = pd.read_csv('./data/2020_S/A3/A3_data.csv')
cr1000_a3_df['date'] = cr1000_a3_df['date'].str.cat(cr1000_a3_df['time'], sep=' ')
cr1000_a3_df.index = cr1000_a3_df['date']
cr1000_a3_df = cr1000_a3_df.drop(['date', 'time'], axis=1)
cr1000_a3_df.index = pd.DatetimeIndex(cr1000_a3_df.index)

In [23]:
cr1000_a3_df = cr1000_a3_df.groupby(cr1000_a3_df.index).mean()
cr1000_a3_df = cr1000_a3_df.reindex(pd.date_range(cr1000_a3_df.index[0], cr1000_a3_df.index[-1], freq='1 min')).interpolate()
cr1000_a3_df['Pyrano_Wsec'] = (cr1000_a3_df.loc[:, 'Pyrano_Wsec_1'] + cr1000_a3_df.loc[:, 'Pyrano_Wsec_2'])/2
cr1000_a3_df = cr1000_a3_df[['Pyrano_Wsec', 'Temp_Avg', 'Humid_Avg']]

In [24]:
SW2_df = pd.concat([_2.loc[pd.date_range(cr1000_a3_df.index[0], _2.index[-1], freq='1 min')],
                    cr1000_a3_df.loc[pd.date_range(cr1000_a3_df.index[0], _2.index[-1], freq='1 min')]], axis=1)

In [25]:
SW2_df

,Loadcell_4,subs_VWC,subs_EC,subs_bulk_EC,subs_temp,permittivity,Pyrano_Wsec,Temp_Avg,Humid_Avg
2020-02-26 00:00:00,-1.30,0.00,2.77,1.45,20.6,45.96,0.0,22.4500,49.100
2020-02-26 00:01:00,-1.30,69.10,2.77,1.45,20.6,45.96,0.0,22.6875,48.725
2020-02-26 00:02:00,-1.30,69.15,2.67,1.40,20.6,46.01,0.0,22.9000,48.350
2020-02-26 00:03:00,-1.30,69.14,2.77,1.45,20.6,46.00,0.0,23.1250,47.875
2020-02-26 00:04:00,-1.30,69.11,2.69,1.41,20.7,45.97,0.0,23.2875,47.475
...,...,...,...,...,...,...,...,...,...
2020-07-09 23:55:00,-8.71,12.39,0.00,0.57,26.1,2.78,0.0,26.9375,76.675
2020-07-09 23:56:00,-8.71,12.42,0.00,0.58,26.1,2.78,0.0,26.9250,76.725
2020-07-09 23:57:00,-8.71,12.40,0.00,0.57,26.1,2.78,0.0,26.9250,76.900
2020-07-09 23:58:00,-8.71,12.41,0.00,0.56,26.0,2.78,0.0,26.9250,76.925


In [26]:
SW2_df.columns = ['loadcell_4', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'rad', 'temp', 'hum']

In [27]:
SW2_df = SW2_df[['temp', 'hum', 'rad', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_4']]

In [28]:
SW2_df.to_csv('./results/2020_S/SW2_FR_greenhouse_origin.csv')

# 2020 Winter

# Control

In [29]:
DIRECTORY = './data/2020_W/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.dat')]
dataset_list.sort()

In [30]:
dataset_list

['CR1000_Table1_2020_09_17_17_30_53.dat',
 'CR1000_Table1_2020_09_18_17_30_53.dat',
 'CR1000_Table1_2020_09_30_17_54_45.dat',
 'CR1000_Table1_2020_10_04_19_57_51.dat',
 'CR1000_Table1_2020_10_09_14_23_58.dat',
 'CR1000_Table1_2020_10_17_12_59_47.dat',
 'CR1000_Table1_2020_11_01_18_18_09.dat',
 'CR1000_Table1_2020_11_06_01_35_18.dat',
 'CR1000_Table1_2020_11_09_11_50_33.dat',
 'CR1000_Table1_2020_11_16_11_23_33.dat',
 'CR1000_Table1_2020_11_27_10_39_34.dat',
 'CR1000_Table1_2020_12_12_13_22_24.dat',
 'CR1000_Table1_2020_12_17_16_35_36.dat',
 'CR1000_Table1_2020_12_28_10_14_20.dat',
 'CR1000_Table1_2021_01_01_15_36_18.dat',
 'CR1000_Table1_2021_01_12_23_38_23.dat',
 'CR1000_Table1_2021_01_18_11_08_00.dat',
 'CR1000_Table1_2021_01_21_16_00_53.dat',
 'CR1000_Table1_2021_02_02_01_56_04.dat']

In [31]:
greenhouse_df = []
for FILENAME in dataset_list:
    try:
        temp_df = pd.read_csv(DIRECTORY + FILENAME, sep='\t', index_col='TIMESTAMP', skiprows=[0, 2, 3])
    except ValueError:
        temp_df = pd.read_csv(DIRECTORY + FILENAME, sep=',', index_col='TIMESTAMP', skiprows=[0, 2, 3])
    greenhouse_df.append(temp_df)
greenhouse_df = pd.concat(greenhouse_df, sort=True)

In [32]:
greenhouse_df.index = pd.DatetimeIndex(greenhouse_df.index)

In [33]:
cr1000_df = greenhouse_df
cr1000_df = cr1000_df[['Loadcell_1', 'Loadcell_2', 'Pyrano_Wsec_1', 'Temp_Avg', 'Humid_Avg']]

### 2020 Winter A3

In [34]:
DIRECTORY = './data/2020_W/A3_CR1000/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.dat')]
dataset_list.sort()

In [35]:
dataset_list

['200424-200916_A3.dat', '200916-201017_A3.dat', '201017-210101_A3.dat']

In [36]:
greenhouse_df = []
for FILENAME in dataset_list:
    try:
        temp_df = pd.read_csv(DIRECTORY + FILENAME, sep='\t', index_col='TIMESTAMP', skiprows=[0, 2, 3])
    except ValueError:
        temp_df = pd.read_csv(DIRECTORY + FILENAME, sep=',', index_col='TIMESTAMP', skiprows=[0, 2, 3])
    greenhouse_df.append(temp_df)
greenhouse_df = pd.concat(greenhouse_df, sort=True)

/home/phil/.virtualenvs/tf20/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3051: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [37]:
greenhouse_df.index = pd.DatetimeIndex(greenhouse_df.index)

In [38]:
greenhouse_df['Pyrano_Wsec'] = (greenhouse_df.loc[:, 'Pyrano_Wsec_1'] + greenhouse_df.loc[:, 'Pyrano_Wsec_2'])/2

In [39]:
cr1000_a3_df = greenhouse_df
cr1000_a3_df = cr1000_a3_df[['Pyrano_Wsec', 'Temp_Avg', 'Humid_Avg']]

#### date range

In [40]:
cr1000_df = cr1000_df.loc['2020-08-26 00:00:00':'2021-01-24 23:59:00']
cr1000_a3_df = cr1000_a3_df.loc['2020-08-26 00:00:00':'2021-01-24 23:59:00']

In [41]:
date_index = pd.date_range(cr1000_df.index[0], cr1000_df.index[-1], freq='1 min')

In [42]:
cr1000_a3_df = cr1000_a3_df.reindex(date_index)

In [43]:
DIRECTORY = './data/2020_W/'
file_list = os.listdir(DIRECTORY)
dataset_list = [file for file in file_list if file.endswith('.xlsx')]
dataset_list.sort()

In [44]:
dataset_list

['SNU_SW_CT_loadcell_20200826-20201023.xlsx',
 'SNU_SW_CT_loadcell_20201024-20201221.xlsx',
 'SNU_SW_CT_loadcell_20201222-20201231.xlsx',
 'SNU_SW_CT_loadcell_20210101-20210124.xlsx',
 'SNU_SW_FR_loadcell_20200826-20200830.xlsx',
 'SNU_SW_FR_loadcell_20200831-20200929.xlsx',
 'SNU_SW_FR_loadcell_20200930-20201030.xlsx',
 'SNU_SW_FR_loadcell_20201031-20201101.xlsx',
 'SNU_SW_FR_loadcell_20201102-20201204.xlsx',
 'SNU_SW_FR_loadcell_20201205-20201231.xlsx',
 'SNU_SW_FR_loadcell_20210101-20210124.xlsx',
 'SNU_SW_RB1_loadcell_20201003-20201023.xlsx',
 'SNU_SW_RB1_loadcell_20201024-20201221.xlsx',
 'SNU_SW_RB1_loadcell_20201222-20201231.xlsx',
 'SNU_SW_RB1_loadcell_20210101-20210124.xlsx',
 'SNU_SW_RB2_loadcell_20201001-20201023.xlsx',
 'SNU_SW_RB2_loadcell_20201024-20201221.xlsx',
 'SNU_SW_RB2_loadcell_20201222-20201231.xlsx',
 'SNU_SW_RB2_loadcell_20210101-20210124.xlsx']

In [45]:
greenhouse_df = []
div_1 = 0
div_2 = 0
div_3 = 0
for FILENAME in dataset_list:
    if 'CT' in FILENAME:
        div_1 += 1
    if 'FR' in FILENAME:
        div_2 += 1
    if 'RB1' in FILENAME:
        continue
    if 'RB2' in FILENAME:
        div_3 += 1
    temp_df = pd.read_excel(DIRECTORY + FILENAME, index_col='date')
    temp_df.index = pd.DatetimeIndex(temp_df.index)
    temp_df = temp_df[['weight', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']]
    greenhouse_df.append(temp_df)

In [46]:
for i in range(len(greenhouse_df)):
    if greenhouse_df[i].index.round('1 min')[0].minute == 1:
        greenhouse_df[i].index = (greenhouse_df[i].index.round('1 min') - pd.Timedelta('1 min'))
    elif greenhouse_df[i].index.round('1 min')[0].minute == 2:
        greenhouse_df[i].index = (greenhouse_df[i].index.round('2 min') - pd.Timedelta('2 min'))
    else:
        greenhouse_df[i].index = greenhouse_df[i].index.round('1 min')

In [47]:
for i in range(len(greenhouse_df)):
    greenhouse_df[i] = greenhouse_df[i].groupby(greenhouse_df[i].index).first()

In [48]:
date_index = pd.date_range(greenhouse_df[0].index[0], greenhouse_df[-1].index[-1], freq='1 min')

In [49]:
other_df = []
for FILENAME in dataset_list:
    if 'RB1' in FILENAME:
        temp_df = pd.read_excel(DIRECTORY + FILENAME, index_col='date')
        temp_df.index = pd.DatetimeIndex(temp_df.index)
        temp_df = temp_df[['weight']]
        other_df.append(temp_df)
    else:
        continue

In [50]:
ct_df = pd.concat(greenhouse_df[:div_1]).reindex(date_index)
fr_df = pd.concat(greenhouse_df[div_1:div_1+div_2]).reindex(date_index)
rb_df = pd.concat(greenhouse_df[div_1+div_2:]).reindex(date_index)

In [51]:
ct_df.columns = ['Loadcell_3', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']

In [52]:
SW2_ct_df = pd.concat([ct_df.loc[pd.date_range(cr1000_df.index[0], ct_df.index[-1], freq='1 min')],
                       cr1000_df.loc[pd.date_range(cr1000_df.index[0], ct_df.index[-1], freq='1 min')]], axis=1)

/home/phil/.virtualenvs/tf20/lib/python3.6/site-packages/ipykernel_launcher.py:2: FutureWarning: 
Passing list-likes to .loc or [] with any missing label will raise
KeyError in the future, you can use .reindex() as an alternative.

See the documentation here:
https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#deprecate-loc-reindex-listlike
  


In [53]:
SW2_ct_df = SW2_ct_df.astype('float')

In [54]:
SW2_ct_df.columns = ['loadcell_3', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_1', 'loadcell_2', 'rad', 'temp', 'hum']

In [55]:
SW2_ct_df = SW2_ct_df[['temp', 'hum', 'rad', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_1', 'loadcell_2', 'loadcell_3']]

In [56]:
SW2_ct_df['temp'] = ((SW2_ct_df['temp']-1000)/80)
SW2_ct_df['hum'] = ((SW2_ct_df['hum']-1000)/40)
SW2_ct_df['loadcell_1'] = ((SW2_ct_df['loadcell_1']/100))
SW2_ct_df['loadcell_2'] = ((SW2_ct_df['loadcell_2']/100))

In [57]:
SW2_ct_df.loc[:, 'loadcell_1'] = SW2_ct_df.loc[:, 'loadcell_1'] +11/100
SW2_ct_df.loc[:, 'loadcell_2'] = SW2_ct_df.loc[:, 'loadcell_2'] -388/100 #calibration
SW2_ct_df.loc[:, 'loadcell_3'] = SW2_ct_df.loc[:, 'loadcell_3']*2 -8.3

In [58]:
SW2_ct_df.to_csv('./results/2020_W/SW_CT_greenhouse_origin.csv')

In [69]:
SW2_ct_df.loc[:, 'loadcell_1'] = SW2_ct_df.loc[:, 'loadcell_1'] - 0.8
SW2_ct_df.loc[:, 'loadcell_2'] = SW2_ct_df.loc[:, 'loadcell_2'] - 0.5
SW2_ct_df.loc[:, 'loadcell_3'] = SW2_ct_df.loc[:, 'loadcell_3'] - 0.8

In [60]:
other_df = pd.concat(other_df).reindex(date_index)

In [61]:
rb_df['weight2'] = other_df.values

In [62]:
fr_df.columns = ['Loadcell_4', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity']
rb_df.columns = ['Loadcell_6', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permittivity', 'Loadcell_5']

In [63]:
SW2_fr_df = pd.concat([fr_df.loc[pd.date_range(cr1000_a3_df.index[0], fr_df.index[-1], freq='1 min')],
                       cr1000_a3_df.loc[pd.date_range(cr1000_a3_df.index[0], fr_df.index[-1], freq='1 min')]], axis=1)
SW2_rb_df = pd.concat([rb_df.loc[pd.date_range(cr1000_a3_df.index[0], rb_df.index[-1], freq='1 min')],
                       cr1000_a3_df.loc[pd.date_range(cr1000_a3_df.index[0], rb_df.index[-1], freq='1 min')]], axis=1)

In [64]:
SW2_fr_df.columns = ['loadcell_4', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'rad', 'temp', 'hum']
SW2_rb_df.columns = ['loadcell_6', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_5', 'rad', 'temp', 'hum']

In [65]:
SW2_fr_df = SW2_fr_df[['temp', 'hum', 'rad', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_4']]
SW2_rb_df = SW2_rb_df[['temp', 'hum', 'rad', 'subs_VWC', 'subs_EC', 'subs_bulk_EC', 'subs_temp', 'permit', 'loadcell_5', 'loadcell_6']]

In [66]:
SW2_fr_df['temp'] = ((SW2_fr_df['temp']-1000)/80)
SW2_fr_df['hum'] = ((SW2_fr_df['hum']-1000)/40)
SW2_fr_df['loadcell_4'] = ((SW2_fr_df['loadcell_4']))

SW2_rb_df['temp'] = ((SW2_rb_df['temp']-1000)/80)
SW2_rb_df['hum'] = ((SW2_rb_df['hum']-1000)/40)
SW2_rb_df['loadcell_5'] = ((SW2_rb_df['loadcell_5']))
SW2_rb_df['loadcell_6'] = ((SW2_rb_df['loadcell_6']))

In [67]:
SW2_fr_df.to_csv('./results/2020_W/SW_FR_greenhouse_origin.csv')
SW2_rb_df.to_csv('./results/2020_W/SW_RB_greenhouse_origin.csv')